In [1]:
"""Utilities for Quad4FHE plaintext experiments.

This module is intentionally small and dependency-light so it can be copied next
into any single-dataset notebook/script. It provides two things:

1. A tee context manager that writes all notebook/script stdout/stderr to a log
   file while still showing it on screen.
2. Direct JSON export from quad4fhe.ReplacementResult objects, avoiding fragile
   regex parsing of copied notebook output.
"""

from __future__ import annotations

import contextlib
import json
import math
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional


class _TeeStream:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return any(getattr(stream, "isatty", lambda: False)() for stream in self.streams)


@contextlib.contextmanager
def tee_stdout_stderr(log_path: str | Path):
    """Duplicate stdout/stderr to ``log_path`` and the current console.

    Use around the whole experiment:

        with tee_stdout_stderr("results/my_run.txt"):
            main()
    """
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    old_stdout, old_stderr = sys.stdout, sys.stderr
    with log_path.open("w", encoding="utf-8", buffering=1) as fh:
        sys.stdout = _TeeStream(old_stdout, fh)
        sys.stderr = _TeeStream(old_stderr, fh)
        try:
            print(f"[autosave] stdout/stderr log -> {log_path}")
            yield log_path
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def to_jsonable(obj: Any) -> Any:
    """Convert common scientific Python objects to strict JSON values."""
    # Local imports keep this helper usable even in minimal environments.
    try:
        import numpy as np
    except Exception:  # pragma: no cover
        np = None
    try:
        import pandas as pd
    except Exception:  # pragma: no cover
        pd = None
    try:
        import torch
    except Exception:  # pragma: no cover
        torch = None

    if obj is None or isinstance(obj, (str, bool, int)):
        return obj

    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None

    if np is not None:
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            value = float(obj)
            return value if math.isfinite(value) else None
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return to_jsonable(obj.tolist())

    if torch is not None and hasattr(torch, "is_tensor") and torch.is_tensor(obj):
        return to_jsonable(obj.detach().cpu().numpy())

    if isinstance(obj, Path):
        return str(obj)

    if pd is not None:
        if isinstance(obj, pd.DataFrame):
            return to_jsonable(obj.to_dict(orient="records"))
        if isinstance(obj, pd.Series):
            return to_jsonable(obj.to_dict())

    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]

    # Last resort: preserve readable values without breaking json.dump.
    return str(obj)


def save_json(obj: Dict[str, Any], path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(to_jsonable(obj), fh, indent=2, ensure_ascii=False, allow_nan=False)
    print(f"[autosave] JSON -> {path}")
    return path


def save_csv(rows: Iterable[Dict[str, Any]], path: str | Path) -> Path:
    import pandas as pd
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(list(rows)).to_csv(path, index=False)
    print(f"[autosave] CSV -> {path}")
    return path


def dataframe_records(df: Any) -> list:
    if df is None:
        return []
    try:
        return to_jsonable(df.to_dict(orient="records"))
    except Exception:
        return []


def dataframe_test_by_method(df: Any) -> Dict[str, Dict[str, Any]]:
    """Return {method: metrics} for rows with Split == 'test'."""
    out: Dict[str, Dict[str, Any]] = {}
    if df is None:
        return out
    try:
        sub = df[df["Split"] == "test"]
        for _, row in sub.iterrows():
            method = str(row.get("Method"))
            metrics = {str(k): to_jsonable(v) for k, v in row.to_dict().items()
                       if k not in ("Method", "Split")}
            out[method] = metrics
    except Exception:
        return out
    return out


def _metric_from_table(table: Dict[str, Dict[str, Any]], method: str, *keys: str) -> Any:
    row = table.get(method, {})
    for key in keys:
        if key in row:
            return row[key]
    return None


def quad_report_diagnostics(result: Any, split: str, n_expected: Optional[int] = None) -> Dict[str, Any]:
    """Extract agreement/mismatch diagnostics for fit/calibration or test split."""
    attr_candidates = []
    if split in ("fit", "calib", "calibration"):
        attr_candidates = ["fit_diagnostics", "calib_diagnostics", "calibration_diagnostics"]
        split_label = "fit"
    else:
        attr_candidates = ["test_diagnostics"]
        split_label = "test"

    diag = None
    for attr in attr_candidates:
        value = getattr(result, attr, None)
        if value is not None:
            diag = dict(value)
            break

    if diag is None:
        diag = {}
        df = getattr(result, "report_vs_pseudo", None)
        if df is not None:
            try:
                row = df[(df["Method"] == "Quad4FHE") & (df["Split"] == split_label)]
                if len(row) > 0:
                    diag["agreement"] = float(row.iloc[0]["Agreement"])
            except Exception:
                pass

    n_value = diag.get("n", diag.get("calib_n", diag.get("n_samples", n_expected)))
    agreement = diag.get("agreement", diag.get("calib_agreement", diag.get("fit_agreement")))
    mismatch = diag.get("mismatch_count", diag.get("calib_mismatch_count", diag.get("fit_mismatch_count")))

    if mismatch is None and agreement is not None and n_value is not None:
        mismatch = int(round((1.0 - float(agreement)) * int(n_value)))

    exact = diag.get("exact_preserved", diag.get("exact_preserved_on_calib", diag.get("fit_exact_preserved")))
    if exact is None and mismatch is not None:
        exact = bool(int(mismatch) == 0)

    return {
        "split": split_label,
        "n": to_jsonable(n_value),
        "agreement": to_jsonable(agreement),
        "mismatch_count": to_jsonable(mismatch),
        "exact_preserved": to_jsonable(exact),
    }


def quad_solver_diagnostics(result: Any) -> Dict[str, Any]:
    return to_jsonable(dict(getattr(result, "solver_diagnostics", None) or {}))


def build_quad4fhe_run_record(
    *,
    result: Any,
    key: str,
    hidden_dim: Optional[int],
    fit_n: Optional[int],
    test_n: Optional[int],
    pool_fraction: Optional[float] = None,
    rep_fit_size: Optional[int] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Build a JSON-serializable record for one Quad4FHE run."""
    fit_diag = quad_report_diagnostics(result, "fit", n_expected=fit_n)
    test_diag = quad_report_diagnostics(result, "test", n_expected=test_n)
    solver_diag = quad_solver_diagnostics(result)

    report_truth_test = dataframe_test_by_method(getattr(result, "report_vs_truth", None))
    report_pseudo_test = dataframe_test_by_method(getattr(result, "report_vs_pseudo", None))

    calib_agreement = fit_diag.get("agreement")
    test_agreement = test_diag.get("agreement")
    gap = None
    if calib_agreement is not None and test_agreement is not None:
        gap = float(calib_agreement) - float(test_agreement)

    # Common keys requested by reviewers. Keep all solver diagnostics too.
    common_solver_keys = [
        "num_pairwise_constraints",
        "min_pairwise_margin",
        "normalized_min_pairwise_margin",
        "slack_positive_count",
        "sum_slack",
        "mean_slack",
        "max_slack",
        "pairwise_slack_positive_count",
        "pairwise_sum_slack",
        "pairwise_mean_slack",
        "pairwise_max_slack",
        "selected_C",
        "soft_C_grid",
        "soft_trace",
        "selected_mu",
        "mu_grid",
        "mu_p",
        "mu_n",
    ]

    quad = {
        "alpha": getattr(result, "alpha", None),
        "beta": getattr(result, "beta", None),
        "eta": getattr(result, "eta", None),
        "threshold": getattr(result, "threshold", None),
        "zero_threshold_realized": getattr(result, "zero_threshold_realized", None),
        "method_used": getattr(result, "method_used", None),
        "hard_feasible": getattr(result, "feasible", None),
        "empirical_margin": getattr(result, "empirical_margin", None),
        "normalized_margin": getattr(result, "normalized_margin", None),
        "quant_decimals": getattr(result, "quant_decimals", None),
        "constraint_version": getattr(result, "constraint_version", None),
        "he_artifact_dir": str(getattr(result, "he_export_dir", None)) if getattr(result, "he_export_dir", None) else None,
        "calib_n": fit_diag.get("n"),
        "calib_agreement": calib_agreement,
        "calib_mismatch_count": fit_diag.get("mismatch_count"),
        "exact_preserved_on_calib": fit_diag.get("exact_preserved"),
        "test_n": test_diag.get("n"),
        "test_agreement": test_agreement,
        "test_mismatch_count": test_diag.get("mismatch_count"),
        "calib_test_agreement_gap": gap,
        "test_top1_acc": _metric_from_table(report_truth_test, "Quad4FHE", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Quad4FHE", "Top5", "Top-5"),
        "test_macro_f1": _metric_from_table(report_truth_test, "Quad4FHE", "MacroF1", "F1"),
        "solver_diagnostics": solver_diag,
    }
    for k in common_solver_keys:
        quad[k] = solver_diag.get(k)

    # Fill a few derived diagnostics that are convenient for tables.
    if quad.get("pairwise_mean_slack") is None:
        denom = quad.get("num_pairwise_constraints")
        num = quad.get("pairwise_sum_slack")
        try:
            if denom not in (None, 0) and num is not None:
                quad["pairwise_mean_slack"] = float(num) / float(denom)
        except Exception:
            pass
    if quad.get("selected_mu") is None:
        method = quad.get("method_used")
        try:
            if isinstance(method, str) and method.startswith("rch") and quad.get("mu_p") == quad.get("mu_n"):
                quad["selected_mu"] = quad.get("mu_p")
        except Exception:
            pass

    original = {
        "test_top1_acc": _metric_from_table(report_truth_test, "Original", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Original", "Top5", "Top-5"),
    }

    record = {
        "key": key,
        "hidden_dim": hidden_dim,
        "pool_fraction": pool_fraction,
        "rep_fit_size": rep_fit_size,
        "original": original,
        "quad4fhe": quad,
        "report_vs_ground_truth_test": report_truth_test,
        "report_vs_pseudo_labels_test": report_pseudo_test,
        "report_vs_ground_truth_records": dataframe_records(getattr(result, "report_vs_truth", None)),
        "report_vs_pseudo_labels_records": dataframe_records(getattr(result, "report_vs_pseudo", None)),
    }
    if extra:
        record.update(to_jsonable(extra))
    return to_jsonable(record)


def make_experiment_payload(
    *,
    dataset: str,
    experiment: str,
    seed: int,
    dataset_info: Dict[str, Any],
    config: Dict[str, Any],
    source_script: Optional[str] = None,
    log_file: Optional[str | Path] = None,
) -> Dict[str, Any]:
    return {
        "dataset": dataset,
        "experiment": experiment,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_script": source_script,
        "log_file": str(log_file) if log_file is not None else None,
        "seed": seed,
        "dataset_info": to_jsonable(dataset_info),
        "config": to_jsonable(config),
        "runs": [],
    }


def write_combined_dataset_json(dataset: str, root_dir: str | Path = "results",
                                output_path: Optional[str | Path] = None) -> Optional[Path]:
    """Merge autosaved fulltrain/smallpool JSONs into one dataset-level JSON.

    This keeps compatibility with the older `<dataset>_results.json` convention.
    Missing halves are skipped, so it is safe to call after either notebook.
    """
    root = Path(root_dir) / dataset
    if output_path is None:
        output_path = root / f"{dataset}_results.json"
    output_path = Path(output_path)

    combined: Dict[str, Any] = {"dataset": dataset, "created_at": datetime.now().isoformat(timespec="seconds")}
    found = False
    for exp in ("fulltrain", "smallpool"):
        p = root / exp / f"{dataset}_{exp}_results.json"
        if not p.exists():
            continue
        with p.open("r", encoding="utf-8") as fh:
            block = json.load(fh)
        combined[exp] = {
            "source_json": str(p),
            "source_script": block.get("source_script"),
            "log_file": block.get("log_file"),
            "dataset_info": block.get("dataset_info", {}),
            "config": block.get("config", {}),
            "runs": block.get("runs", []),
        }
        if exp == "fulltrain":
            combined[exp]["cross_hidden_dim_summary"] = block.get("cross_hidden_dim_summary", [])
        if exp == "smallpool":
            combined[exp]["cross_pool_tables"] = block.get("cross_pool_tables", {})
            combined[exp]["meta_table"] = block.get("meta_table", [])
        found = True

    if not found:
        print(f"[autosave] no fulltrain/smallpool JSON found under {root}; combined JSON not written")
        return None
    return save_json(combined, output_path)


import traceback


In [2]:
import os
import gc
import sys
import copy
import time
import random
import hashlib
import warnings
import faulthandler
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, List

faulthandler.enable()
warnings.filterwarnings("ignore")

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

import gzip
import json

try:
    from huggingface_hub import hf_hub_download
    HAS_HF_HUB = True
except Exception:
    hf_hub_download = None
    HAS_HF_HUB = False

try:
    from transformers import AutoTokenizer, AutoModel
    HAS_TRANSFORMERS = True
except Exception:
    AutoTokenizer = None
    AutoModel = None
    HAS_TRANSFORMERS = False

import quad4fhe


# ============================================================
# Configuration
# ============================================================
SEED = 2026

# ---------- MASSIVE-specific ----------
# NOTE: the original AmazonScience/massive repo ships a legacy Python loader
# script (massive.py) which is no longer supported by datasets>=4.0.0
# ("RuntimeError: Dataset scripts are no longer supported"). We instead pull
# from the mteb parquet/JSONL mirrors, which expose the same records via
# plain file downloads through huggingface_hub.
MASSIVE_LABEL_FIELD = "intent"    # "intent" (60 classes) or "scenario" (18 classes)
NUM_CLASSES         = 60          # 60 for intent, 18 for scenario

HF_DATASET_NAME = ("mteb/amazon_massive_intent" if MASSIVE_LABEL_FIELD == "intent"
                   else "mteb/amazon_massive_scenario")
DATASET_NAME = "Qwen3_massive"
EXPERIMENT_NAME = "fulltrain"

# mteb mirrors use SHORT locale codes (en, de, fr, ...) instead of the BCP-47
# style (en-US, de-DE) used by the original AmazonScience repo. zh-CN stays.
DEFAULT_CONFIGS = ["en", "de", "fr", "es", "ja", "zh-CN"]

DEFAULT_INSTRUCTION = (
    "Classify the intent of the given user utterance."
    if MASSIVE_LABEL_FIELD == "intent" else
    "Classify the scenario of the given user utterance."
)
# --------------------------------------

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MAX_LENGTH = 128

HIDDEN_DIMS = [64, 128, 256]

EPOCHS = 100
LR = 1e-3
BATCH_SIZE_TRAIN = 256
BATCH_SIZE_EVAL = 1024
WEIGHT_DECAY = 1e-5
PATIENCE = 10
PRINT_EVERY = 5

BATCH_SIZE_EMBED = 16
USE_AMP       = True
NORMALIZE_EMB = True              # L2-normalise Qwen3 embeddings (standard)

# --- Paths ---
FEATURE_CACHE_DIR = "./data"                                           # Qwen3 embedding cache dir
OUTPUT_DIR = Path("..") / "results" / DATASET_NAME / EXPERIMENT_NAME
LOG_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_result.txt"
JSON_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_results.json"
SUMMARY_CSV_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_summary.csv"

BASELINES = ["square", "ls_poly_2", "ls_poly_3", "ls_poly_5", "ls_poly_7",
             "remez_2", "remez_3", "remez_5", "remez_7",
             "ola", "precise_a11"]

# Populated at runtime from the dataset's ClassLabel feature.
MASSIVE_CLASS_NAMES: Dict[int, str] = {}


# ============================================================
# Utilities
# ============================================================
def separator(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    sys.stdout.flush()


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


# ============================================================
# MASSIVE loading via mteb mirrors (hf_hub_download + gzipped JSONL)
# ============================================================
def _massive_download_split(repo_id: str, split: str, lang: str) -> str:
    """Download {split}/{lang}.json.gz from an mteb MASSIVE mirror to the HF cache.

    `split` is one of 'train' / 'validation' / 'test'. Files are cached by
    huggingface_hub under ~/.cache/huggingface/hub, so re-runs are instant.
    """
    if not HAS_HF_HUB:
        raise ImportError("huggingface_hub is required. `pip install -U huggingface_hub`")
    return hf_hub_download(
        repo_id=repo_id,
        filename=f"{split}/{lang}.json.gz",
        repo_type="dataset",
    )


def _read_gz_jsonl(path: str):
    """Yield dict records from a gzipped JSON Lines file."""
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def _massive_read_records(records, split_tag: str, label_field: str):
    """Read (texts, raw_labels) from an iterable of dicts.

    `raw_labels` is kept in its original type (int or str) and converted later
    by the caller, once a global label->int mapping is known.
    The mteb mirror uses 'text' + 'label' (string class name);
    raw MASSIVE uses 'utt' + 'intent'/'scenario' (may be int or str).
    """
    texts: List[str] = []
    raw_labels: List = []
    for r in records:
        # --- text ---
        t = r.get("text", r.get("utt", r.get("sentence")))
        if t is None:
            raise KeyError(f"MASSIVE {split_tag}: no text field. Keys={list(r.keys())}")
        # --- label (may be int or string) ---
        lv = r.get("label")
        if lv is None:
            lv = r.get(label_field)
        if lv is None:
            raise KeyError(
                f"MASSIVE {split_tag}: no label field. Expected 'label' or "
                f"'{label_field}'. Keys={list(r.keys())}"
            )
        texts.append(str(t))
        raw_labels.append(lv)
    return texts, raw_labels


def _build_label_mapping(all_raw_labels, label_field: str):
    """Given a flat list of raw label values (all int OR all str), return
    (label_to_idx, class_names_dict).

    If values are already ints, returns (None, {}) -- caller should just cast
    to int64. Otherwise builds a deterministic str -> int mapping (sorted
    alphabetically) so train/val/test/locales all agree on the label IDs.
    """
    if not all_raw_labels:
        return None, {}
    if all(isinstance(v, (int, np.integer)) for v in all_raw_labels):
        return None, {}
    # String labels: build stable sorted mapping.
    unique = sorted({str(v) for v in all_raw_labels})
    label_to_idx = {s: i for i, s in enumerate(unique)}
    class_names = {i: s for s, i in label_to_idx.items()}
    if len(unique) != NUM_CLASSES:
        print(f"  WARNING: NUM_CLASSES={NUM_CLASSES} but dataset yielded "
              f"{len(unique)} '{label_field}' classes. Update NUM_CLASSES to match.")
    print(f"  Built label mapping with {len(unique)} '{label_field}' classes "
          f"(string -> int, alphabetical order)")
    return label_to_idx, class_names


def _to_int_labels(raw_labels, label_to_idx) -> np.ndarray:
    if label_to_idx is None:
        return np.asarray([int(v) for v in raw_labels], dtype=np.int64)
    return np.asarray([label_to_idx[str(v)] for v in raw_labels], dtype=np.int64)


def load_massive_multilingual(configs=DEFAULT_CONFIGS, dataset_name=HF_DATASET_NAME,
                                label_field=MASSIVE_LABEL_FIELD):
    """Load MASSIVE (via the mteb mirror) per-locale, concatenate same-split
    across locales. Returns dict with train_net / val_net / test / train_all.

    Two-step design:
      1. Walk every (locale, split) file, keeping labels in their raw form.
      2. Build a single global str->int mapping (if labels are strings) and
         convert everything, so train/val/test and all locales share IDs.
    """
    global MASSIVE_CLASS_NAMES
    SPLIT_MAP = [("train_net", "train"), ("val_net", "validation"), ("test", "test")]

    raw_store = {k: {"texts": [], "raw_labels": []} for k in ["train_net", "val_net", "test"]}

    for cfg in configs:
        n_log = {}
        for split_name, orig_split in SPLIT_MAP:
            local_path = _massive_download_split(dataset_name, orig_split, cfg)
            texts, raw_labels = _massive_read_records(
                _read_gz_jsonl(local_path), f"{cfg}/{orig_split}", label_field)
            raw_store[split_name]["texts"].extend(texts)
            raw_store[split_name]["raw_labels"].extend(raw_labels)
            n_log[split_name] = len(texts)
        print(f"  [{cfg}] train={n_log['train_net']:6d}  "
              f"val={n_log['val_net']:5d}  test={n_log['test']:5d}")

    # Build one global label mapping from every label we saw (across splits/locales).
    all_raw = (raw_store["train_net"]["raw_labels"]
               + raw_store["val_net"]["raw_labels"]
               + raw_store["test"]["raw_labels"])
    label_to_idx, class_names = _build_label_mapping(all_raw, label_field)
    if class_names:
        MASSIVE_CLASS_NAMES = class_names

    out = {}
    for k in ["train_net", "val_net", "test"]:
        labels = _to_int_labels(raw_store[k]["raw_labels"], label_to_idx)
        out[k] = (raw_store[k]["texts"], labels)

    out["train_all"] = (
        out["train_net"][0] + out["val_net"][0],
        np.concatenate([out["train_net"][1], out["val_net"][1]], axis=0).astype(np.int64),
    )
    return out


# ============================================================
# Qwen3 embedding extraction (last-token pooling + optional instruction)
# ============================================================
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f"Instruct: {task_description}\nQuery:{query}"


def last_token_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Pool the final non-pad token. Assumes left-padding (standard for Qwen3-Embedding)."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    batch = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch, device=last_hidden_states.device), seq_lens]


def _autocast_context(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


@torch.no_grad()
def embed_texts_qwen3(texts, model_name, device, batch_size=BATCH_SIZE_EMBED,
                       max_length=MAX_LENGTH, instruction=DEFAULT_INSTRUCTION,
                       normalize=NORMALIZE_EMB, amp=USE_AMP):
    if not HAS_TRANSFORMERS:
        raise ImportError("transformers>=4.51.0 is required. `pip install transformers>=4.51.0`")

    print(f"Embedding {len(texts)} texts with {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {}
    if device.type == "cuda":
        model_kwargs["torch_dtype"] = torch.float16
    model = AutoModel.from_pretrained(model_name, **model_kwargs).to(device)
    model.eval()

    use_amp = amp and device.type == "cuda"
    outputs_all = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    for b, start in enumerate(range(0, len(texts), batch_size), start=1):
        batch_texts = texts[start:start + batch_size]
        if instruction:
            batch_texts = [get_detailed_instruct(instruction, t) for t in batch_texts]
        batch = tokenizer(batch_texts, padding=True, truncation=True,
                           max_length=max_length, return_tensors="pt")
        batch = {k: v.to(device) for k, v in batch.items()}
        with _autocast_context(device, use_amp):
            out = model(**batch)
            emb = last_token_pool(out.last_hidden_state, batch["attention_mask"])
            if normalize:
                emb = F.normalize(emb, p=2, dim=1)
        outputs_all.append(emb.detach().cpu().float().numpy())
        if b == 1 or b % 50 == 0 or b == n_batches:
            print(f"  Embed batch {b:>5d}/{n_batches}")

    del model, tokenizer
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("  Qwen3 backbone released from memory.")
    return np.concatenate(outputs_all, axis=0).astype(np.float32)


def feature_cache_path(cache_dir, model_name, configs, max_length, instruction, label_field):
    os.makedirs(cache_dir, exist_ok=True)
    tag = "__".join(configs)
    digest = hashlib.md5((tag + "|" + str(max_length) + "|" + str(instruction or "") + "|" + label_field)
                          .encode("utf-8")).hexdigest()[:10]
    model_tag = model_name.split("/")[-1].replace("-", "_")
    return os.path.join(cache_dir, f"{model_tag}_massive_{label_field}_{digest}_fulltrain_features.npz")


def ensure_qwen3_feature_cache(splits_raw, model_name, cache_path, device,
                                batch_size=BATCH_SIZE_EMBED, max_length=MAX_LENGTH,
                                instruction=DEFAULT_INSTRUCTION, normalize=NORMALIZE_EMB,
                                amp=USE_AMP):
    """Compute embeddings for train_net / val_net / test separately (caching all of them)."""
    required = ["train_net_features", "train_net_labels",
                "val_net_features",   "val_net_labels",
                "test_features",      "test_labels"]
    if os.path.exists(cache_path):
        print(f"Loading cached features: {cache_path}")
        pack = np.load(cache_path, allow_pickle=False)
        if all(k in pack for k in required):
            if (pack["train_net_features"].shape[0] == len(splits_raw["train_net"][0])
                and pack["val_net_features"].shape[0] == len(splits_raw["val_net"][0])
                and pack["test_features"].shape[0] == len(splits_raw["test"][0])):
                out = {
                    "train_net": (pack["train_net_features"].astype(np.float32),
                                  pack["train_net_labels"].astype(np.int64)),
                    "val_net":   (pack["val_net_features"].astype(np.float32),
                                  pack["val_net_labels"].astype(np.int64)),
                    "test":      (pack["test_features"].astype(np.float32),
                                  pack["test_labels"].astype(np.int64)),
                }
                out["train_all"] = (
                    np.concatenate([out["train_net"][0], out["val_net"][0]], axis=0).astype(np.float32),
                    np.concatenate([out["train_net"][1], out["val_net"][1]], axis=0).astype(np.int64),
                )
                print("  Cache OK.")
                return out
        print("  Cache mismatch or missing keys. Recomputing features...")

    t0 = time.perf_counter()
    feats = {}
    for split_name in ["train_net", "val_net", "test"]:
        separator(f"Embedding split: {split_name}  (n={len(splits_raw[split_name][0])})")
        tx, ly = splits_raw[split_name]
        fx = embed_texts_qwen3(tx, model_name, device,
                                batch_size=batch_size, max_length=max_length,
                                instruction=instruction, normalize=normalize, amp=amp)
        feats[split_name] = (fx, ly)
    print(f"Total embedding time: {time.perf_counter() - t0:.1f}s")

    np.savez(
        cache_path,
        train_net_features=feats["train_net"][0], train_net_labels=feats["train_net"][1],
        val_net_features=feats["val_net"][0],     val_net_labels=feats["val_net"][1],
        test_features=feats["test"][0],           test_labels=feats["test"][1],
    )
    print(f"Saved feature cache: {cache_path}")

    feats["train_all"] = (
        np.concatenate([feats["train_net"][0], feats["val_net"][0]], axis=0).astype(np.float32),
        np.concatenate([feats["train_net"][1], feats["val_net"][1]], axis=0).astype(np.int64),
    )
    return feats


# ============================================================
# Classifier head + training
# ============================================================
class FeatureClassifier(nn.Module):
    """Linear -> ReLU -> Linear head on top of frozen Qwen3 embeddings."""
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-5):
        self.patience = patience; self.min_delta = min_delta
        self.best = float("inf"); self.counter = 0; self.best_state = None

    def step(self, loss, model):
        if loss < self.best - self.min_delta:
            self.best = loss; self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def train_head(X_tr, y_tr, X_va, y_va, input_dim, hidden_dim, num_classes, device):
    model = FeatureClassifier(input_dim, hidden_dim, num_classes).to(device)
    tr_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                                          torch.tensor(y_tr, dtype=torch.long)),
                           batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.tensor(X_va, dtype=torch.float32),
                                          torch.tensor(y_va, dtype=torch.long)),
                           batch_size=BATCH_SIZE_EVAL, shuffle=False)

    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    stopper = EarlyStopping()
    amp_enabled = USE_AMP and device.type == "cuda"
    scaler = torch.amp.GradScaler(device="cuda", enabled=amp_enabled)

    print(f"  Architecture: Frozen Qwen3-Embedding-0.6B({input_dim}) -> "
          f"Linear({input_dim}->{hidden_dim}) -> ReLU -> Linear({hidden_dim}->{num_classes})")
    print(f"  Optimizer: Adam, lr={LR}, wd={WEIGHT_DECAY}, epochs={EPOCHS}, batch={BATCH_SIZE_TRAIN}")
    print(f"  Device: {device}, AMP: {amp_enabled}")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        for xb, yb in tr_loader:
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb); loss = ce(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total_loss += loss.item() * xb.size(0); total_n += xb.size(0)
        train_loss = total_loss / max(total_n, 1)

        model.eval()
        v_loss, v_n, v_preds, v_true = 0.0, 0, [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    logits = model(xb); loss = ce(logits, yb)
                v_loss += loss.item() * xb.size(0); v_n += xb.size(0)
                v_preds.append(torch.argmax(logits, 1).cpu().numpy())
                v_true.append(yb.cpu().numpy())
        val_loss = v_loss / max(v_n, 1)
        val_acc = accuracy_score(np.concatenate(v_true), np.concatenate(v_preds))

        if epoch == 1 or epoch % PRINT_EVERY == 0 or epoch == EPOCHS:
            print(f"    Epoch {epoch:>3d}/{EPOCHS} | train_loss={train_loss:.6f} | "
                  f"val_loss={val_loss:.6f} | val_acc={val_acc:.4f}")

        if stopper.step(val_loss, model):
            print(f"    Early stopping at epoch {epoch}, best_val_loss={stopper.best:.6f}")
            break

    stopper.restore(model)
    model.cpu().eval()
    return model


# ============================================================
# Main
# ============================================================
def main():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not HAS_TRANSFORMERS:
        raise ImportError("This script requires transformers>=4.51.0")
    if not HAS_HF_HUB:
        raise ImportError("This script requires huggingface_hub. `pip install -U huggingface_hub`")

    # ---------------- Step 1: Load MASSIVE ----------------
    separator("Step 1: Load MASSIVE (official splits, merged across locales)")
    print(f"Dataset       : {DATASET_NAME}")
    print(f"HF dataset    : {HF_DATASET_NAME}")
    print(f"Configs       : {DEFAULT_CONFIGS}")
    print(f"Label field   : '{MASSIVE_LABEL_FIELD}'")
    print(f"Num classes   : {NUM_CLASSES}")
    print(f"Model         : {MODEL_NAME}")
    print(f"HIDDEN_DIMS   = {HIDDEN_DIMS}")
    splits_raw = load_massive_multilingual(DEFAULT_CONFIGS, HF_DATASET_NAME, MASSIVE_LABEL_FIELD)
    for key in ["train_net", "val_net", "train_all", "test"]:
        texts_i, y_i = splits_raw[key]
        vals = np.unique(y_i)
        print(f"  {key:10s}: n={len(texts_i):7d}, classes_present={len(vals)}/{NUM_CLASSES}")

    # Quick sanity check on NUM_CLASSES vs what we actually saw
    n_classes_seen = len(np.unique(splits_raw["train_all"][1]))
    if n_classes_seen != NUM_CLASSES:
        print(f"\n  WARNING: train_all has {n_classes_seen} classes but NUM_CLASSES={NUM_CLASSES}")

    # ---------------- Step 2: Qwen3 embeddings ----------------
    separator("Step 2: Extract / load frozen Qwen3 embeddings")
    cache_path = feature_cache_path(FEATURE_CACHE_DIR, MODEL_NAME, DEFAULT_CONFIGS,
                                     MAX_LENGTH, DEFAULT_INSTRUCTION, MASSIVE_LABEL_FIELD)
    feats = ensure_qwen3_feature_cache(splits_raw, MODEL_NAME, cache_path, device)

    X_train = feats["train_net"][0]; y_train = feats["train_net"][1]
    X_val   = feats["val_net"][0];   y_val   = feats["val_net"][1]
    X_tall  = feats["train_all"][0]; y_tall  = feats["train_all"][1]
    X_test  = feats["test"][0];      y_test  = feats["test"][1]
    print(f"  Feature shapes: train_net={X_train.shape}, val={X_val.shape}, "
          f"train_all={X_tall.shape}, test={X_test.shape}")

    # Fit StandardScaler on train only (no leakage from val/test)
    scaler = StandardScaler().fit(X_train)
    X_train_std = scaler.transform(X_train).astype(np.float64)
    X_val_std   = scaler.transform(X_val).astype(np.float64)
    X_tall_std  = scaler.transform(X_tall).astype(np.float64)
    X_test_std  = scaler.transform(X_test).astype(np.float64)
    input_dim = X_train_std.shape[1]
    print(f"  Qwen3 embedding dim = {input_dim}")

    # ---------------- Step 3: Loop over hidden_dim ----------------
    all_summaries = []

    payload = make_experiment_payload(
        dataset=DATASET_NAME,
        experiment=EXPERIMENT_NAME,
        seed=SEED,
        dataset_info={
            "input_dim": int(input_dim),
            "num_classes": int(NUM_CLASSES),
            "splits": {
                "source": "official_splits_merged_across_locales",
                "train_net": int(len(X_train_std)),
                "val_net": int(len(X_val_std)),
                "train_all": int(len(X_tall_std)),
                "test": int(len(X_test_std)),
            },
            "hf_dataset": HF_DATASET_NAME,
            "embedding_model": MODEL_NAME,
            "embedding_normalized": NORMALIZE_EMB,
            "feature_cache_dir": FEATURE_CACHE_DIR,
            "frozen_backbone": True,
            "encrypted_component": "classification_head_only",
        },
        config={
            "hidden_dims": HIDDEN_DIMS,
            "epochs": EPOCHS,
            "lr": LR,
            "batch_size_train": BATCH_SIZE_TRAIN,
            "batch_size_eval": BATCH_SIZE_EVAL,
            "batch_size_embed": BATCH_SIZE_EMBED,
            "weight_decay": WEIGHT_DECAY,
            "patience": PATIENCE,
            "max_length": MAX_LENGTH,
            "use_amp": USE_AMP,
            "normalize_emb": NORMALIZE_EMB,
            "default_instruction": DEFAULT_INSTRUCTION,
            "default_configs": globals().get("DEFAULT_CONFIGS"),
            "massive_label_field": globals().get("MASSIVE_LABEL_FIELD"),
            "val_fraction_of_train": globals().get("VAL_FRACTION_OF_TRAIN"),
            "baselines": BASELINES,
        },
        source_script=f"{DATASET_NAME}_{EXPERIMENT_NAME}_autosave.ipynb",
        log_file=LOG_PATH,
    )
    for hd in HIDDEN_DIMS:
        separator(f"[hidden_dim={hd}] Step 3: Train classifier head")
        set_seed(SEED)
        model = train_head(X_train_std, y_train, X_val_std, y_val,
                           input_dim, hd, NUM_CLASSES, device)

        # Test top-1 (top-5 is meaningful for 60-class intent; trivial for 18-class scenario)
        with torch.no_grad():
            orig_test_logits = model(torch.tensor(X_test_std, dtype=torch.float32)).cpu().numpy()
        orig_top1 = float(np.mean(orig_test_logits.argmax(axis=1) == y_test))
        print(f"  Original head test top-1 ACC = {orig_top1:.4f}")

        # ---------------- Step 4: quad4fhe.replace() ----------------
        separator(f"[hidden_dim={hd}] Step 4: quad4fhe.replace(task='multiclass', ...)")
        X_fit = X_tall_std; y_fit = y_tall
        print(f"  fit_data: n={len(X_fit)} (train + val)")
        print(f"  test_data: n={len(X_test_std)}")
        # MASSIVE-intent with 6 locales gives n_fit ~ 78k and K=60, so pairwise
        # constraints ~ n_fit * (K-1) ~ 4.6M. This exceeds the library's
        # max_exact_constraints (200k), so cutting-plane activates automatically.

        result = quad4fhe.replace(
            task              = "multiclass",
            model             = model,
            hidden_layer      = "fc1",
            output_layer      = "fc2",
            activation        = "relu",
            fit_data          = (X_fit, y_fit),
            test_data         = (X_test_std, y_test),
            baselines         = BASELINES,
            export_he_dir     = f"he_artifacts_qwen3_massive_{MASSIVE_LABEL_FIELD}_h{hd}",
            use_cutting_plane = True,
            use_osqp_warmstart= False,
            verbose           = True,
            seed              = SEED,
        )

        rm = result.replaced_model.eval()
        with torch.no_grad():
            logits_quad = rm(torch.tensor(X_test_std, dtype=torch.float32)).cpu().numpy()
        quad_top1 = float(np.mean(logits_quad.argmax(axis=1) == y_test))
        print(f"\n  Quad4FHE test top-1 ACC = {quad_top1:.4f}")

        # Save per-hd summary CSV
        hd_dir = os.path.join(OUTPUT_DIR, f"hidden_{hd}")
        os.makedirs(hd_dir, exist_ok=True)
        if result.report_vs_truth is not None:
            result.report_vs_truth.to_csv(os.path.join(hd_dir, "report_vs_truth.csv"), index=False)
            print(f"  Saved report: {hd_dir}/report_vs_truth.csv")

        summary = {
            "hidden_dim":        hd,
            "label_field":       MASSIVE_LABEL_FIELD,
            "method_used":       result.method_used,
            "hard_feasible":     result.feasible,
            "alpha":             result.alpha,
            "beta":              result.beta,
            "eta":               result.eta,
            "empirical_margin":  result.empirical_margin,
            "normalized_margin": result.normalized_margin,
            "quant_decimals":    result.quant_decimals,
            "orig_top1":         orig_top1,
            "quad_top1":         quad_top1,
            "he_export_dir":     result.he_export_dir,
        }
        all_summaries.append(summary)

        summary = all_summaries[-1]
        fit_diag = quad_report_diagnostics(result, "fit", n_expected=len(X_fit))
        test_diag = quad_report_diagnostics(result, "test", n_expected=len(X_test_std))
        solver_diag = quad_solver_diagnostics(result)
        summary.update({
            "calib_n": fit_diag.get("n"),
            "calib_agreement": fit_diag.get("agreement"),
            "calib_mismatch_count": fit_diag.get("mismatch_count"),
            "exact_preserved_on_calib": fit_diag.get("exact_preserved"),
            "test_agreement": test_diag.get("agreement"),
            "test_mismatch_count": test_diag.get("mismatch_count"),
            "calib_test_agreement_gap": (None if fit_diag.get("agreement") is None or test_diag.get("agreement") is None
                                         else fit_diag.get("agreement") - test_diag.get("agreement")),
            "num_pairwise_constraints": solver_diag.get("num_pairwise_constraints"),
            "min_pairwise_margin": solver_diag.get("min_pairwise_margin"),
            "normalized_min_pairwise_margin": solver_diag.get("normalized_min_pairwise_margin"),
            "slack_positive_count": solver_diag.get("slack_positive_count"),
            "sum_slack": solver_diag.get("sum_slack"),
            "mean_slack": solver_diag.get("mean_slack"),
            "max_slack": solver_diag.get("max_slack"),
            "pairwise_slack_positive_count": solver_diag.get("pairwise_slack_positive_count"),
            "pairwise_sum_slack": solver_diag.get("pairwise_sum_slack"),
            "pairwise_mean_slack": (None if solver_diag.get("pairwise_sum_slack") is None or solver_diag.get("num_pairwise_constraints") in (None, 0)
                                    else solver_diag.get("pairwise_sum_slack") / solver_diag.get("num_pairwise_constraints")),
            "pairwise_max_slack": solver_diag.get("pairwise_max_slack"),
            "selected_C": solver_diag.get("selected_C"),
            "selected_mu": solver_diag.get("selected_mu"),
            "constraint_version": getattr(result, "constraint_version", None),
        })
        payload["runs"].append(build_quad4fhe_run_record(
            result=result,
            key=f"hidden_dim={hd}",
            hidden_dim=hd,
            fit_n=len(X_fit),
            test_n=len(X_test_std),
            pool_fraction=None,
            rep_fit_size=None,
        ))
        payload["cross_hidden_dim_summary"] = all_summaries
        save_json(payload, JSON_PATH)
        save_csv(all_summaries, SUMMARY_CSV_PATH)
        print(f"  calibration agreement={fit_diag.get('agreement')} "
              f"mismatches={fit_diag.get('mismatch_count')} "
              f"exact_preserved={fit_diag.get('exact_preserved')}")
        print(f"  test agreement={test_diag.get('agreement')} "
              f"mismatches={test_diag.get('mismatch_count')}")

    # ---------------- Step 5: Cross-hidden-dim summary ----------------
    separator("Step 5: Cross-hidden-dim summary")
    df = pd.DataFrame(all_summaries)
    with pd.option_context("display.float_format", "{:.6f}".format, "display.max_columns", None):
        print(df.to_string(index=False))
    summary_csv = os.path.join(OUTPUT_DIR, "summary_all_hidden_dims.csv")
    df.to_csv(summary_csv, index=False)
    print(f"\nSaved combined summary: {summary_csv}")

    payload["cross_hidden_dim_summary"] = all_summaries
    save_json(payload, JSON_PATH)
    save_csv(all_summaries, SUMMARY_CSV_PATH)
    write_combined_dataset_json(DATASET_NAME, root_dir=OUTPUT_DIR.parent.parent)

    separator("Done")


if __name__ == "__main__":
    try:
        with tee_stdout_stderr(LOG_PATH):
            main()
    except Exception:
        traceback.print_exc()
        sys.stdout.flush(); sys.stderr.flush()
        raise


[autosave] stdout/stderr log -> ../results/Qwen3_massive/fulltrain/Qwen3_massive_fulltrain_result.txt

Step 1: Load MASSIVE (official splits, merged across locales)
Dataset       : Qwen3_massive
HF dataset    : mteb/amazon_massive_intent
Configs       : ['en', 'de', 'fr', 'es', 'ja', 'zh-CN']
Label field   : 'intent'
Num classes   : 60
Model         : Qwen/Qwen3-Embedding-0.6B
HIDDEN_DIMS   = [64, 128, 256]
  [en] train= 11514  val= 2033  test= 2974
  [de] train= 11514  val= 2033  test= 2974
  [fr] train= 11514  val= 2033  test= 2974
  [es] train= 11514  val= 2033  test= 2974
  [ja] train= 11514  val= 2033  test= 2974
  [zh-CN] train= 11514  val= 2033  test= 2974
  Built label mapping with 60 'intent' classes (string -> int, alphabetical order)
  train_net : n=  69084, classes_present=60/60
  val_net   : n=  12198, classes_present=59/60
  train_all : n=  81282, classes_present=60/60
  test      : n=  17844, classes_present=59/60

Step 2: Extract / load frozen Qwen3 embeddings

Embeddin

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Embed batch     1/4318
  Embed batch    50/4318
  Embed batch   100/4318
  Embed batch   150/4318
  Embed batch   200/4318
  Embed batch   250/4318
  Embed batch   300/4318
  Embed batch   350/4318
  Embed batch   400/4318
  Embed batch   450/4318
  Embed batch   500/4318
  Embed batch   550/4318
  Embed batch   600/4318
  Embed batch   650/4318
  Embed batch   700/4318
  Embed batch   750/4318
  Embed batch   800/4318
  Embed batch   850/4318
  Embed batch   900/4318
  Embed batch   950/4318
  Embed batch  1000/4318
  Embed batch  1050/4318
  Embed batch  1100/4318
  Embed batch  1150/4318
  Embed batch  1200/4318
  Embed batch  1250/4318
  Embed batch  1300/4318
  Embed batch  1350/4318
  Embed batch  1400/4318
  Embed batch  1450/4318
  Embed batch  1500/4318
  Embed batch  1550/4318
  Embed batch  1600/4318
  Embed batch  1650/4318
  Embed batch  1700/4318
  Embed batch  1750/4318
  Embed batch  1800/4318
  Embed batch  1850/4318
  Embed batch  1900/4318
  Embed batch  1950/4318


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Embed batch     1/763
  Embed batch    50/763
  Embed batch   100/763
  Embed batch   150/763
  Embed batch   200/763
  Embed batch   250/763
  Embed batch   300/763
  Embed batch   350/763
  Embed batch   400/763
  Embed batch   450/763
  Embed batch   500/763
  Embed batch   550/763
  Embed batch   600/763
  Embed batch   650/763
  Embed batch   700/763
  Embed batch   750/763
  Embed batch   763/763
  Qwen3 backbone released from memory.

Embedding split: test  (n=17844)
Embedding 17844 texts with Qwen/Qwen3-Embedding-0.6B ...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Embed batch     1/1116
  Embed batch    50/1116
  Embed batch   100/1116
  Embed batch   150/1116
  Embed batch   200/1116
  Embed batch   250/1116
  Embed batch   300/1116
  Embed batch   350/1116
  Embed batch   400/1116
  Embed batch   450/1116
  Embed batch   500/1116
  Embed batch   550/1116
  Embed batch   600/1116
  Embed batch   650/1116
  Embed batch   700/1116
  Embed batch   750/1116
  Embed batch   800/1116
  Embed batch   850/1116
  Embed batch   900/1116
  Embed batch   950/1116
  Embed batch  1000/1116
  Embed batch  1050/1116
  Embed batch  1100/1116
  Embed batch  1116/1116
  Qwen3 backbone released from memory.
Total embedding time: 162.1s
Saved feature cache: ./data/Qwen3_Embedding_0.6B_massive_intent_8bb5ba5500_fulltrain_features.npz
  Feature shapes: train_net=(69084, 1024), val=(12198, 1024), train_all=(81282, 1024), test=(17844, 1024)
  Qwen3 embedding dim = 1024

[hidden_dim=64] Step 3: Train classifier head
  Architecture: Frozen Qwen3-Embedding-0.6B(1024) ->